# Jina Reranker Structure tr?n Colab

Notebook n?y ch?y `src/re-ranker/jina_rerank_structure.py` v?i model `jinaai/jina-reranker-v2-base-multilingual` tr?n input structure.


In [ ]:
import torch

print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("Chưa bật GPU")

In [ ]:
!pip uninstall -y transformers tokenizers
!pip install -q "transformers==4.44.2" "tokenizers==0.19.1" accelerate sentence-transformers einops

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
%cd /content

!rm -rf Text-Mining---RAG-on-News
!git clone -b Alibaba https://github.com/TiiAyyLuvBear/Text-Mining---RAG-on-News.git

%cd /content/Text-Mining---RAG-on-News

In [ ]:
import torch
import transformers
import importlib.util

print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

print("transformers:", transformers.__version__)
print("Jina reranker dependencies ready")

In [ ]:
from pathlib import Path

input_path = Path("/content/drive/MyDrive/out_embedding/per_query_structured.jsonl")

print("Input exists:", input_path.exists())
print("Input path:", input_path)

if input_path.exists():
    with input_path.open("r", encoding="utf-8") as f:
        first_line = f.readline()
    print(first_line[:500])

In [ ]:
from pathlib import Path

script_path = Path("src/re-ranker/jina_rerank_structure.py")

print("Script exists:", script_path.exists())
print("Script path:", script_path)


In [ ]:
from pathlib import Path

output_dir = Path("/content/drive/MyDrive/out_reranker")
output_dir.mkdir(parents=True, exist_ok=True)

print("Output dir exists:", output_dir.exists())

## Hugging Face

Model Jina public th??ng kh?ng c?n login. N?u g?p l?i gated/private, ch?y `from huggingface_hub import login; login()`.


In [ ]:
from huggingface_hub import login

login("hf_..........")

In [ ]:
!python src/re-ranker/jina_rerank_structure.py \
    --input "/content/drive/MyDrive/out_embedding/per_query_structured.jsonl" \
    --output "/content/drive/MyDrive/out_reranker/rerank_structure_jina_top5_test.jsonl" \
    --limit 5 \
    --batch-size 8 \
    --max-length 1024

In [ ]:
import json
from pathlib import Path

test_output = Path("/content/drive/MyDrive/out_reranker/rerank_structure_jina_top5_test.jsonl")

print("Test output exists:", test_output.exists())

with test_output.open("r", encoding="utf-8") as f:
    row = json.loads(next(f))

print("QA ID:", row["qa_id"])
print("Question:", row["question"])
print("Embedding type:", row["embedding_type"])
print("Reranker:", row["reranker"])
print("Metrics:", row["rerank_metrics"])
print("Top 1 score:", row["reranked_candidates"][0]["rerank_score"])
print("Top 1 text:", row["reranked_candidates"][0]["text"][:300])

In [ ]:
!python src/re-ranker/jina_rerank_structure.py \
    --input "/content/drive/MyDrive/out_embedding/per_query_structured.jsonl" \
    --output "/content/drive/MyDrive/out_reranker/rerank_structure_jina_top5.jsonl" \
    --batch-size 8 \
    --max-length 1024

In [ ]:
from pathlib import Path

input_file = Path("/content/drive/MyDrive/out_embedding/per_query_structured.jsonl")
output_file = Path("/content/drive/MyDrive/out_reranker/rerank_structure_jina_top5.jsonl")

def count_jsonl(path):
    with path.open("r", encoding="utf-8") as f:
        return sum(1 for line in f if line.strip())

print("Input lines:", count_jsonl(input_file))
print("Output lines:", count_jsonl(output_file))

In [ ]:
import json
import pandas as pd
from pathlib import Path

output_file = Path("/content/drive/MyDrive/out_reranker/rerank_structure_jina_top5.jsonl")
summary_file = Path("/content/drive/MyDrive/out_reranker/rerank_structure_jina_top5_summary.csv")

rows = []

with output_file.open("r", encoding="utf-8") as f:
    for line in f:
        row = json.loads(line)
        rows.append(row["rerank_metrics"])

df = pd.DataFrame(rows)

summary = {
    "config": "structure_jina_reranker",
    "num_queries": len(df),
    "hit@1": df["hit@1"].mean(),
    "hit@5": df["hit@5"].mean(),
    "recall@5": df["recall@5"].mean(),
    "mrr@5": df["mrr@5"].mean(),
    "ndcg@5": df["ndcg@5"].mean(),
}

summary_df = pd.DataFrame([summary])


summary_df.to_csv(summary_file, index=False, encoding="utf-8-sig")
print(f"Saved summary to {summary_file}")
summary_df